# Exploring a 3Di results mesh with pyramids-mesh

This notebook opens a **real** 3Di hydrodynamic simulation result and explores it end to end with
`pyramids-mesh` and `pyramids-core`. 3Di stores its state on a staggered sub-grid rather than an explicit
polygon mesh; the pyramids-mesh 3Di driver is a *needs-mesh* driver that **reconstructs the 2D cell grid**
from the file's geometry arrays and presents it as a standard `MeshDataset`.

Unlike a bare geometry mesh, a results file carries the solver's output fields - water level, velocity,
volume, forcings - so the interesting part here is the *data riding on the grid*, not just the grid.

(3Di files can also carry a coupled 1D network; pyramids represents the 2D mesh and warns that the 1D part
is dropped.)

## Setup

`MeshDataset` is the single entry point. Paths are `pathlib.Path` objects resolved **relative to this
notebook** (the kernel's working directory is the notebook's own folder) - never `str()` a path for a
pyramids API. This notebook lives one level deeper than the format tour, so the data root has one extra
`../`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from pyramids.netcdf.ugrid import MeshDataset
import matplotlib.pyplot as plt
from cleopatra.basemap.tiles import add_tiles

DATA = Path("../../../examples/data/real-data")
DATA.resolve()

## Read the results file

Pass `driver="3di"` explicitly. The driver rebuilds the 2D grid and loads every `Mesh2D_*` result variable.
The `UserWarning` about a coupled 1D network is expected for this file: pyramids keeps the 2D mesh and
reports that the 1D part cannot be represented alongside it, rather than mangling the two together.

In [ ]:
ds = MeshDataset.read_file(DATA / "3di" / "results_3di.nc", driver="3di")
ds

The repr already tells the story: a 2D mesh of several thousand faces carrying ten solver fields. Before
plotting anything, let cleopatra pick the right matplotlib backend. Calling this once, right after the first
read, makes figures embed inline in the notebook and fall back to a headless backend in CI.

In [ ]:
from cleopatra.config import Config
Config.set_matplotlib_backend()

## The mesh at a glance

A `MeshDataset` wraps a UGRID-style unstructured mesh plus the data variables defined on it. The scalar
properties below describe the grid: how many nodes and faces it has, its coordinate reference system, and
its spatial extent.

In [ ]:
print("n_node:", ds.n_node)
print("n_face:", ds.n_face)
print("crs:   ", ds.crs)
print("epsg:  ", ds.epsg)
print("bounds:", ds.bounds)  # (xmin, ymin, xmax, ymax)

Note the mismatch: `crs`/`epsg` are `None`, yet the `bounds` are clearly a **projected grid in metres** -
coordinates around 106 000-111 000 east, 515 000-520 000 north, i.e. the Dutch RD (Rijksdriehoek) national
grid. Many hydraulic-model result files store coordinates without an embedded CRS, so pyramids reports the
geometry faithfully and leaves georeferencing to you. The practical consequence shows up later: the
`if ds.epsg:` guards around basemap tiles all skip, because there is no declared CRS to project tiles into.

## The solver fields

`data_variable_names` lists every field attached to the mesh. This is what makes a *results* file worth
reading - water surface level, velocity components, cell volume, and the forcing/exchange terms the solver
tracks.

In [ ]:
ds.data_variable_names

The key fields are `Mesh2D_s1` (water surface level), `Mesh2D_ucx`/`Mesh2D_ucy` (velocity components),
`Mesh2D_vol` (cell water volume) and `Mesh2D_su` (wet sub-grid storage area). The rest are forcing and
exchange terms (`Mesh2D_rain`, `Mesh2D_q_lat`, `Mesh2D_q_sss`, `Mesh2D_leak`, `Mesh2D_intercepted_volume`).

`summary()` returns a tidy `pandas.DataFrame` of per-variable statistics - a quick sanity check on
magnitudes and fill values across the whole run.

In [ ]:
ds.summary()

Two things to read off the table. First, several fields bottom out at `-9999`: that is 3Di's **dry-cell
fill value**, not a real measurement, and it dominates the statistics wherever cells are dry (it is why the
`Mesh2D_s1` mean looks like a huge negative number). We will mask it before mapping. Second, `Mesh2D_su` and
`Mesh2D_vol` have genuine positive ranges - those are the spatially varying fields with real structure to
look at. The `count` is faces x timesteps, so each field spans the whole run.

## Wireframe - the bare mesh

`plot_outline()` draws the mesh edges with no data colouring. It is the fastest way to see the grid's
shape, resolution, and how faces are distributed before looking at any results. The tile guard skips
(no CRS), so we see the reconstructed grid on its own.

In [ ]:
_g = ds.plot_outline()
if ds.epsg:
    add_tiles(_g.ax, crs=ds.epsg)

The reconstructed grid is a dense, irregular block of quad cells covering a roughly 4.8 km x 5.0 km patch -
a real 3Di model domain, far richer than the handful of cells in a toy file.

## Mapping the results

The result variables carry a time axis. `time_values` lists the stored timesteps (in seconds); we select one
with `isel(time=...)` before mapping. We take a mid-run step so water is present.

In [ ]:
print("timesteps (s):", [round(t, 1) for t in ds.time_values])
step = ds.isel(time=4)  # t = 240 s

### Water surface level - the wetted extent

`masked_values` reads a variable as a plain array; the dry cells still carry the `-9999` fill, so we build a
boolean condition and hand it to `where(...)`, which keeps the wet cells and masks the rest to NaN. Plotting
the masked dataset then colours only the cells that actually hold water.

In [ ]:
s1 = np.asarray(step.masked_values("Mesh2D_s1"))
wet = step.where(s1 > -9000, variable_name="Mesh2D_s1")
_g = wet.plot("Mesh2D_s1", cmap="Blues", colorbar=True,
              title="Water surface level s1 at t=240 s")
if ds.epsg:
    add_tiles(_g.ax, crs=ds.epsg)

The map shows the model's **wetted footprint**: about half the faces hold water, the rest are masked out as
dry. The colour is essentially uniform because the free surface is flat - at this stage the water sits as a
still pool at roughly -4.97 m RD, with no appreciable gradient across the domain. So `s1` here is most
useful as a *where is there water* map rather than a *how deep* map.

### Cell water volume - the field that evolves

`Mesh2D_vol` is the water volume stored in each cell. Dry cells are simply `0` (no fill value), so it maps
directly. Unlike the flat water level, volume varies cell to cell and grows through the run.

In [ ]:
_g = step.plot("Mesh2D_vol", cmap="viridis", colorbar=True,
               title="Cell water volume at t=240 s")
if ds.epsg:
    add_tiles(_g.ax, crs=ds.epsg)

Now there is real spatial structure: volume concentrates where cells are larger or deeper, and the pattern
tracks the wetted region from the previous map. This is the field to watch frame by frame - its maximum
climbs steadily from one timestep to the next as the pool fills.

### Sub-grid storage area - the cell footprints

`Mesh2D_su` is the wet sub-grid surface/storage area of each cell. It is defined on every face and is
static over the run, so it reads as the *geometry* of the sub-grid: how much surface each reconstructed
cell represents.

In [ ]:
_g = step.plot("Mesh2D_su", cmap="cividis", colorbar=True,
               title="Sub-grid storage area su at t=240 s")
if ds.epsg:
    add_tiles(_g.ax, crs=ds.epsg)

The brightest cells are the ones that aggregate the most sub-grid area - the staggered-grid detail that 3Di
keeps below the computational cell. Reading this alongside the volume map explains why some cells hold far
more water than their neighbours: they simply cover more ground.

### Velocity - a quiescent run

The task names velocity as a field to inspect, so we check it honestly. Rather than force a map of a flat
field, we reduce `ucx`/`ucy` to a speed magnitude and tabulate it per timestep - a `pandas` table is the
right tool when the interesting result is *a single number per step*.

In [ ]:
rows = []
for t, sec in enumerate(ds.time_values):
    st = ds.isel(time=t)
    ux = np.asarray(st.masked_values("Mesh2D_ucx"))
    uy = np.asarray(st.masked_values("Mesh2D_ucy"))
    m = (ux > -9000) & (uy > -9000)
    mag = np.hypot(ux[m], uy[m])
    rows.append({"t (s)": round(float(sec), 1),
                 "wet cells": int(m.sum()),
                 "max |u| (m/s)": float(mag.max()),
                 "mean |u| (m/s)": float(mag.mean())})
pd.DataFrame(rows).set_index("t (s)")

The velocity is **identically zero** across every wet cell and every timestep: this is a hydrostatic
filling scenario with no lateral flow, not a propagating flood wave. That is a genuine property of the
dataset, and worth stating rather than hiding behind a blank velocity map.

## GIS for free

A `MeshDataset` is not a dead end - it converts into the vector types the rest of a GIS workflow expects.
Everything below is reached through pyramids; this package only supplies the file-format driver that built
the mesh.

### Faces as vector polygons

`to_geodataframe(variable_name=...)` turns every face into a polygon row carrying that variable's value - a
standard GeoPandas `GeoDataFrame` you can join, filter, or export. We attach the volume field.

In [ ]:
gdf = step.to_geodataframe(variable_name="Mesh2D_vol")
print("rows:", len(gdf), " columns:", list(gdf.columns), " crs:", gdf.crs)
gdf.head()

One polygon per face, in the file's own RD metre coordinates (and `crs=None`, mirroring the dataset).
Plotting that `GeoDataFrame` straight through GeoPandas gives a choropleth of the same values, now as
independent vector features you could write to a shapefile or GeoPackage.

In [ ]:
ax = gdf.plot(column="Mesh2D_vol", cmap="viridis", legend=True, figsize=(6, 5))
if ds.epsg:
    add_tiles(ax, crs=ds.epsg)
plt.title("Mesh2D_vol - per-face polygons")

### The mesh outline as one polygon

`bounding_polygon()` returns a single shapely polygon tracing the mesh's outer boundary - the model's
footprint, useful as a clip mask or extent reference.

In [ ]:
poly = ds.bounding_polygon()
import geopandas as gpd
ax = gpd.GeoSeries([poly]).plot(facecolor="none", edgecolor="crimson", figsize=(5, 5))
if ds.epsg:
    add_tiles(ax, crs=ds.epsg)
plt.title("Mesh bounding polygon")

The outline traces the same ~4.8 x 5.0 km envelope the `bounds` reported, in RD metres.

### Cropping to a sub-region

`subset_by_bounds(xmin, ymin, xmax, ymax)` returns a new, smaller `MeshDataset` holding only the faces
inside the box - the same object type, so every operation above still applies. We derive a box from the
mesh envelope so it works on any extent.

In [ ]:
xmin, ymin, xmax, ymax = ds.bounds
mx, my = (xmin + xmax) / 2, (ymin + ymax) / 2
sub = ds.subset_by_bounds(xmin, ymin, mx, my)
print("faces in full mesh:", ds.n_face, " -> in subset:", sub.n_face)
_g = sub.isel(time=4).plot("Mesh2D_vol", cmap="viridis", colorbar=True,
                           title="Lower-left quadrant - cell volume")
if ds.epsg:
    add_tiles(_g.ax, crs=ds.epsg)

The subset is a fully independent `MeshDataset` with roughly a quarter of the faces, still carrying all the
solver fields - so you can zoom into a region and keep mapping, sampling, or exporting exactly as before.

## What we cannot do here

`to_crs(...)` needs a *source* CRS to project from. This 3Di result stores coordinates without one, so
`ds.crs` is `None` even though the numbers are clearly RD metres. With no declared source CRS, reprojection
is undefined, and so is draping web-map tiles under the plots (hence every `if ds.epsg:` guard skipped). The
fix is to assign the known CRS first (RD New is EPSG:28992) once you have confirmed it, after which tiles and
`to_crs` both work.

> **Note - xarray interop.** pyramids-core exposes a `to_xarray()` bridge, but these notebooks deliberately
> stay within pyramids-core's own GDAL-based UGRID I/O and do not import xarray.

## Takeaway

A real 3Di result is a *needs-mesh* case: the file never stored an explicit polygon mesh, yet the driver
reconstructs the 2D grid and serves it as an ordinary `MeshDataset` with all ten solver fields attached.
From there the workflow is pure pyramids - mask the `-9999` dry fill, select a timestep with `isel`, map
water level, volume and storage, convert faces to vectors, or crop to a sub-region. Reading the data
honestly matters as much as plotting it: here the free surface is a flat still pool and the flow field is
quiescent, so volume and sub-grid area - not water level or velocity - are the fields that carry the
spatial story.